# JAX Data Pipelines for High-Dimensional ML

This notebook extends your basics notebook and focuses on practical data handling for:
- Tabular datasets (rows x columns)
- Images (N, H, W, C)
- Text (N, T) with padding/masking

It also shows where JAX should be used vs when external tooling is better.


## Should preprocessing be done in JAX or PyTorch/other tools?

Short answer:
- Use **JAX** for transformations that should be JIT-compiled and run on accelerator in the training step.
- Use **dataset tools** (NumPy/Pandas/TFDS/HuggingFace Datasets/PyTorch DataLoader) for IO-heavy and CPU preprocessing.

Practical split:
1. IO + parsing + caching + tokenization: outside JAX (CPU pipelines)
2. Batch-time numeric transforms (normalize, crop, mixup, masking, padding logic): JAX-friendly
3. Model forward/backward/update: JAX

This hybrid approach is what most production JAX training stacks use.


In [ ]:
import jax
import jax.numpy as jnp
from jax import random, jit, vmap
import numpy as np
import math

print('JAX version:', jax.__version__)
print('Device:', jax.devices()[0])


## Section 1: Core Pattern - Pure Batch Functions

The key idea is to make each batch transform a pure function:
`batch_out = transform(key, batch_in)`

Then compose with `jit` and `vmap`.


In [ ]:
def normalize_batch(x, mean, std, eps=1e-6):
    return (x - mean) / (std + eps)

x = jnp.array([[1., 2., 3.], [4., 5., 6.]])
mean = jnp.mean(x, axis=0, keepdims=True)
std = jnp.std(x, axis=0, keepdims=True)
print(normalize_batch(x, mean, std))


## Section 2: Tabular Data (N x D)


In [ ]:
# Synthetic tabular data: binary classification
key = random.PRNGKey(0)
N, D = 4096, 64
key, k1, k2 = random.split(key, 3)
X = random.normal(k1, (N, D))
true_w = random.normal(k2, (D,))
logits = X @ true_w
y = (jax.nn.sigmoid(logits) > 0.5).astype(jnp.float32)

print('X shape:', X.shape, 'y shape:', y.shape)


In [ ]:
def train_val_split(X, y, val_frac=0.2):
    n = X.shape[0]
    n_val = int(n * val_frac)
    return (X[:-n_val], y[:-n_val]), (X[-n_val:], y[-n_val:])

(X_train, y_train), (X_val, y_val) = train_val_split(X, y)
print(X_train.shape, X_val.shape)


In [ ]:
def make_batches(key, X, y, batch_size=128, shuffle=True):
    n = X.shape[0]
    idx = jnp.arange(n)
    if shuffle:
        idx = random.permutation(key, idx)
    for start in range(0, n - batch_size + 1, batch_size):
        b = idx[start:start+batch_size]
        yield X[b], y[b]

key, k = random.split(key)
xb, yb = next(make_batches(k, X_train, y_train, 256))
print(xb.shape, yb.shape)


### Exercise 1
Implement a `standardize_from_train(X_train, X_other)` function that computes mean/std from train only and transforms `X_other`.


In [ ]:
# Exercise cell

def standardize_from_train(X_train, X_other, eps=1e-6):
    # TODO
    mean = jnp.mean(X_train, axis=0, keepdims=True)
    std = jnp.std(X_train, axis=0, keepdims=True)
    return (X_other - mean) / (std + eps)

X_val_std = standardize_from_train(X_train, X_val)
print('val mean (approx):', jnp.mean(X_val_std, axis=0)[:5])


## Section 3: Image Data (N x H x W x C)


In [ ]:
# Fake image batch (for demonstration)
key, k = random.split(key)
N_img, H, W, C = 128, 32, 32, 3
images = random.uniform(k, (N_img, H, W, C))
print(images.shape)


In [ ]:
def random_horizontal_flip(key, imgs, p=0.5):
    n = imgs.shape[0]
    flip_mask = random.bernoulli(key, p=p, shape=(n, 1, 1, 1))
    flipped = imgs[:, :, ::-1, :]
    return jnp.where(flip_mask, flipped, imgs)

@jit
def center_crop(imgs, crop_h=28, crop_w=28):
    h, w = imgs.shape[1], imgs.shape[2]
    sh = (h - crop_h) // 2
    sw = (w - crop_w) // 2
    return imgs[:, sh:sh+crop_h, sw:sw+crop_w, :]

key, k = random.split(key)
aug = random_horizontal_flip(k, images)
cropped = center_crop(aug)
print('aug:', aug.shape, 'cropped:', cropped.shape)


In [ ]:
@jit
def resize_nearest(imgs, new_h, new_w):
    # lightweight educational resize: nearest-neighbor
    n, h, w, c = imgs.shape
    ys = jnp.floor(jnp.linspace(0, h - 1, new_h)).astype(jnp.int32)
    xs = jnp.floor(jnp.linspace(0, w - 1, new_w)).astype(jnp.int32)
    return imgs[:, ys[:, None], xs[None, :], :]

small = resize_nearest(cropped, 16, 16)
print(small.shape)


### Exercise 2
Add random brightness jitter: `imgs + uniform(-delta, delta)` clipped to `[0, 1]`.


In [ ]:
# Exercise solution

def random_brightness(key, imgs, delta=0.1):
    noise = random.uniform(key, (imgs.shape[0], 1, 1, 1), minval=-delta, maxval=delta)
    return jnp.clip(imgs + noise, 0.0, 1.0)

key, k = random.split(key)
out = random_brightness(k, images)
print(out.min(), out.max())


## Section 4: Text Data (N x T)

For text, tokenization is usually outside JAX. Inside JAX you handle:
- padding/truncation
- attention masks
- batch shaping


In [ ]:
PAD_ID = 0
samples = [
    [10, 11, 12, 13],
    [7, 8],
    [5, 6, 7],
    [20, 21, 22, 23, 24, 25],
]

def pad_to_len(seq, T, pad_id=PAD_ID):
    seq = seq[:T]
    return seq + [pad_id] * (T - len(seq))

T = 6
tokens = jnp.array([pad_to_len(s, T) for s in samples], dtype=jnp.int32)
mask = (tokens != PAD_ID).astype(jnp.float32)
print('tokens:
', tokens)
print('mask:
', mask)


In [ ]:
def make_next_token_targets(tokens, pad_id=PAD_ID):
    x = tokens[:, :-1]
    y = tokens[:, 1:]
    loss_mask = (y != pad_id).astype(jnp.float32)
    return x, y, loss_mask

x_tok, y_tok, loss_mask = make_next_token_targets(tokens)
print(x_tok.shape, y_tok.shape, loss_mask.shape)


### Exercise 3
Given `tokens` and `PAD_ID`, compute a causal + padding attention mask with shape `(B, 1, T, T)`.


In [ ]:
# Exercise solution

def make_causal_padding_mask(tokens, pad_id=0):
    B, T = tokens.shape
    valid = (tokens != pad_id).astype(jnp.float32)
    pad_mask = valid[:, None, None, :]                    # (B,1,1,T)
    causal = jnp.tril(jnp.ones((T, T), dtype=jnp.float32)) # (T,T)
    return pad_mask * causal[None, None, :, :]            # (B,1,T,T)

attn_mask = make_causal_padding_mask(tokens)
print(attn_mask.shape)


## Section 5: End-to-End Mini Example (Tabular MLP)


In [ ]:
from flax import linen as nn
import optax

class TabMLP(nn.Module):
    hidden: int = 128

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden)(x)
        x = nn.relu(x)
        x = nn.Dense(1)(x)
        return x.squeeze(-1)

model = TabMLP(hidden=64)
key, k = random.split(key)
params = model.init(k, X_train[:8])
opt = optax.adam(1e-3)
opt_state = opt.init(params)


In [ ]:
@jit
def bce_logits_loss(logits, targets):
    return jnp.mean(optax.sigmoid_binary_cross_entropy(logits, targets))

@jit
def train_step(params, opt_state, xb, yb):
    def loss_fn(p):
        logits = model.apply(p, xb)
        return bce_logits_loss(logits, yb)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = opt.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


In [ ]:
key, k = random.split(key)
for step, (xb, yb) in enumerate(make_batches(k, X_train, y_train, batch_size=256, shuffle=True)):
    params, opt_state, loss = train_step(params, opt_state, xb, yb)
    if step % 5 == 0:
        print(f'step={step:02d} loss={float(loss):.4f}')
    if step == 20:
        break


## Section 6: Production Tips

1. Keep data loader outside JAX; keep per-batch math inside JAX.
2. `jit` stable-shape functions only (avoid dynamic shape recompilation).
3. For multi-device, pre-shard batches and use `pmap`/`pjit`.
4. Cache expensive CPU transforms and tokenize once.
5. In notebooks, avoid `pip install` mid-session unless necessary; restart kernel after dependency changes.
